In [ ]:
from IPython.display import display, HTML, clear_output
display(HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Bricolage+Grotesque:opsz,wght@12..96,800&family=Be+Vietnam+Pro:wght@400;500;600&display=swap');
:root{--bg:#e9f0ee;--panel:#fbfdfc;--ink:#0d2b2e;--mu:#547074;--ln:#cbdad7;--teal:#0e7c7b;--coral:#e4572e;--tint:#d6e8e4}
@media (prefers-color-scheme:dark){:root{--bg:#0a1517;--panel:#112024;--ink:#e3f0ee;--mu:#8fadb0;--ln:#233a3e;--teal:#3cc7c0;--coral:#ff7a4d;--tint:#16302f}}
body,.jp-Notebook,.jp-OutputArea,.voila-body,#voila-body{background:var(--bg)!important;color:var(--ink)!important;font-family:'Be Vietnam Pro',system-ui,sans-serif!important}
.app{max-width:1000px;margin:0 auto;padding:12px 8px}
.app h1{font:800 clamp(2rem,5vw,3.2rem)/1.05 'Bricolage Grotesque',system-ui,sans-serif;letter-spacing:-.025em;margin:8px 0 12px;max-width:16ch;color:var(--ink)}
.app h2{font:800 1.3rem 'Bricolage Grotesque',system-ui,sans-serif;margin:0 0 4px;color:var(--ink)}
.app p{color:var(--mu);margin:0 0 12px}
.panel{background:var(--panel);border:1px solid var(--ln);border-radius:20px;padding:22px;margin:0 14px 16px 0;flex:1 1 420px;min-width:300px}
.tint{background:var(--tint)}
.cols{display:flex;flex-wrap:wrap}
.widget-label,.widget-readout{color:var(--mu)!important}
.widget-text input{background:var(--panel)!important;color:var(--ink)!important;border:1.5px solid var(--ln)!important;border-radius:10px!important;font-weight:600!important}
.widget-button{border-radius:10px!important;font-weight:600!important;margin-top:6px}
.num{font:800 3rem/1 'Bricolage Grotesque',system-ui,sans-serif;letter-spacing:-.03em;color:var(--ink)}
.num small{font:500 .9rem 'Be Vietnam Pro',sans-serif;color:var(--mu);margin-left:6px}
.pill{display:inline-block;margin-top:8px;padding:3px 12px;border-radius:99px;font-weight:600;font-size:.85rem;color:#0d2b2e}
.scale{position:relative;height:12px;border-radius:6px;margin:18px 0 6px;display:flex}.scale i{display:block}
.mk{position:absolute;top:-6px;width:4px;height:24px;border-radius:2px;background:var(--ink);transform:translateX(-2px)}.mk.c{background:var(--coral)}
.ticks{display:flex;justify-content:space-between;font-size:.72rem;color:var(--mu)}
.warn{color:var(--coral);font-size:.85rem;margin-top:8px}
</style>
<div class="app"><h1>Cholesterol theo tuổi và BMI</h1>
<p>Hai công cụ: tính BMI từ chiều cao và cân nặng, rồi dự đoán cholesterol bằng mô hình hồi quy tuyến tính (tuổi + BMI) huấn luyện từ <code>health.csv</code>.</p></div>
"""))

In [1]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.linear_model import LinearRegression

COL_AGE, COL_BMI, COL_CHOL = "age", "bmi", "chol"

health = pd.read_csv("health.csv")[[COL_AGE, COL_BMI, COL_CHOL]].dropna()
model_both = LinearRegression().fit(health[[COL_AGE, COL_BMI]], health[COL_CHOL])

def tinh_bmi(chieu_cao_cm, can_nang_kg):
    return can_nang_kg / (chieu_cao_cm / 100) ** 2      # đổi cm sang m

def phan_loai_bmi(bmi):
    """Trả về (nhóm, mức nguy cơ) theo bảng của đề."""
    if bmi < 18.5: return "Underweight (thiếu cân)", "Thấp"
    if bmi < 25:   return "Normal (bình thường)", "Trung bình"
    if bmi < 30:   return "Overweight (thừa cân)", "Tăng nhẹ"
    if bmi < 35:   return "Obese class I", "Vừa"
    if bmi < 40:   return "Obese class II", "Nặng"
    return "Obese class III", "Rất nặng"

In [2]:
w_cao = widgets.FloatText(value=170, description="Cao (cm)")
w_nang = widgets.FloatText(value=65, description="Nặng (kg)")
nut_bmi = widgets.Button(description="Tính BMI", button_style="success")
out_bmi = widgets.Output()

def khi_bam_bmi(_):
    with out_bmi:
        clear_output(wait=True)
        if w_cao.value <= 0 or w_nang.value <= 0:
            print("Chiều cao và cân nặng phải lớn hơn 0.")
            return
        bmi = tinh_bmi(w_cao.value, w_nang.value)
        nhom, nguy_co = phan_loai_bmi(bmi)
        print(f"BMI = {bmi:.1f} kg/m² | Nhóm: {nhom} | Nguy cơ sức khỏe: {nguy_co}")

nut_bmi.on_click(khi_bam_bmi)
display(widgets.VBox([widgets.HBox([w_cao, w_nang, nut_bmi]), out_bmi]))

In [3]:
w_cao2 = widgets.FloatText(value=170, description="Cao (cm)")
w_nang2 = widgets.FloatText(value=65, description="Nặng (kg)")
w_tuoi = widgets.IntSlider(value=35, min=18, max=90, description="Tuổi")
nut_du_doan = widgets.Button(description="Dự đoán", button_style="info")
out_du_doan = widgets.Output()

def khi_bam_du_doan(_):
    with out_du_doan:
        clear_output(wait=True)
        if w_cao2.value <= 0 or w_nang2.value <= 0:
            print("Chiều cao và cân nặng phải lớn hơn 0.")
            return
        bmi = tinh_bmi(w_cao2.value, w_nang2.value)
        x_moi = pd.DataFrame([[w_tuoi.value, bmi]], columns=[COL_AGE, COL_BMI])  # đúng tên cột lúc huấn luyện
        chol = float(model_both.predict(x_moi)[0])
        print(f"BMI tính được: {bmi:.1f}")
        print(f"Cholesterol dự đoán: {chol:.2f}")
        if not (health[COL_AGE].min() <= w_tuoi.value <= health[COL_AGE].max()
                and health[COL_BMI].min() <= bmi <= health[COL_BMI].max()):
            print("Lưu ý: đầu vào nằm ngoài khoảng dữ liệu huấn luyện, kết quả kém tin cậy.")

nut_du_doan.on_click(khi_bam_du_doan)
display(widgets.VBox([widgets.HBox([w_cao2, w_nang2, w_tuoi, nut_du_doan]), out_du_doan]))